In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

using Shakespeare dateset to train the model


In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-10-19 16:31:37--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.02s   

2026-10-19 16:31:37 (46.9 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
with open("input.txt", "r", encoding="utf-8") as f:
  text = f.read()
len(text)

1115394

here we consider each char as our tokens there are other ways to define our tokens. Some examples:
<br/>
https://github.com/google/sentencepiece
<br/>
https://github.com/openai/tiktoken


In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)


In [ ]:
# mapping from characters to numbers
stoi = {char:num for num, char in enumerate(chars)}
itos = {num:char for num, char in enumerate(chars)}
encode = lambda string : list((stoi[char] for char in string))
decode = lambda code : "".join(list((itos[i] for i in code)))


encode all the training set and store it as tensor

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)

torch.Size([1115394]) torch.int64


In [ ]:
# split data into train and test set.portion of 9/1

n = int(0.9*len(data))
train_data = data[:n]
test_data = data[n:]

In [ ]:
# example of how the whole process works
block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size +1]
for i in range(block_size):
  context = x[:i+1]
  target = y[i]
  print(f"when input is {context} the next desired word: {target}")

when input is tensor([18]) the next desired word: 47
when input is tensor([18, 47]) the next desired word: 56
when input is tensor([18, 47, 56]) the next desired word: 57
when input is tensor([18, 47, 56, 57]) the next desired word: 58
when input is tensor([18, 47, 56, 57, 58]) the next desired word: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the next desired word: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the next desired word: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the next desired word: 58


Here we are selecting {{batch_size}} random starting position, for each batch we will create sequence of 8 input tokens and the 8 outputs related to each subsequence


In [ ]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8
num_embd_dim = 32

def get_batch(split):
  if split == "train":
    data = train_data
  else:
    data = test_data
  # generate {{batch_size}} numbers from zero to data size - block size
  ix = torch.randint(len(data)-block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x,y

xb, yb = get_batch('train')
print(xb)
print(yb)

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


In [ ]:
# BigramLanguageModel
torch.manual_seed(1337)
class BigramLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
    # each token directly reads off the logits for the next token from a lookup table
    self.token_embedding_table = nn.Embedding(vocab_size,num_embd_dim)
    self.position_embedding_table = nn.Embedding(block_size, num_embd_dim)
    self.lm_head = nn.Linear(num_embd_dim, vocab_size)


  def forward(self, idx, targets=None):
    B, T = idx.shape

    token_emb = self.token_embedding_table(idx) #(batch,T(sequence),Channels(here vocab_size))
    pos_emb = self.position_embedding_table(T) # (T,C)
    x = token_emb + pos_emb # (B,T,C)
    logits = self.lm_head(token_emb) #(B,T,vocab_size)
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T,C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, idx, max_new_tokens):
    # idx is (B, T) array of indices in the current context
    for _ in range(max_new_tokens):
      logits, loss = self(idx) #get the prediction
      logits = logits[:, -1, :] # becomes (B, C)
      probs = F.softmax(logits, dim=-1) # getting probabilities
      idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
      idx = torch.cat((idx, idx_next), dim=1) #(B, T+1)
    return idx

model = BigramLanguageModel()
out, loss = model(xb, yb)
print(out.shape)
print(loss)

torch.Size([32, 65])
tensor(4.3962, grad_fn=<NllLossBackward0>)


In [ ]:
# create an optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [ ]:
batch_size = 32
for _ in range(10000):
  xb, yb = get_batch('train')
  logits, loss = model(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

print(loss.item())

2.394822597503662


In [ ]:
print(decode(model.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


LA c wo the;
Pancalolinghowhatharean:
QA:

Wwhass bowoond;
Fomere d shdeenotep.
CI y mbotot swefesea


we want to make tokens have impact on each other but different tokens should have different impact on a specific token so we use two vectors for each tokens first one is key and the other one is query , for calculating he weight of each token we multiply key and query.token in position n should communicate with all the previous tokens. The easiest way is to calc average of all previous channels.
<br>
To do that we will use triangle matrix and multiple it with our matrix.


In [ ]:
torch.manual_seed(1337)
B, T, C = 4,8,2
x = torch.randn(B,T,C)
weight = torch.tril(torch.ones(T, T))
weight = weight / weight.sum(1, keepdim=True)
bag_of_word = weight @ x # for each batch we calculate
bag_of_word

tensor([[[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]],

        [[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]],

        [[-0.6631, -0.2513],
         [ 0.1735, -0.0649],
         [ 0.1685,  0.3348],
         [-0.1621,  0.1765],
         [-0.2312, -0.0436],
         [-0.1015, -0.2855],
         [-0.2593, -0.1630],
         [-0.3015, -0.2293]],

        [[ 1.6455, -0.8030],
         [ 1.4985, -0.5395],
         [ 0.4954,  0.3420],
         [ 1.0623, -0.1802],
         [ 1.1401, -0.4462],
         [ 1.0870, -0.4071],
         [ 1.0430, -0.1299],
         [ 1.1138, -0.1641]]])